# RadarML Lite — Track 2: coherent frequency-domain learning

**Goal:** improve the old spectrogram CNN's low-SNR performance by keeping the complex
Fourier phase information. This classifier learns one mean waveform per supplied
training configuration, then computes frequency-domain matched-filter likelihoods.
It is a fitted statistical model, rather than an epoch-based neural network.
There are **no pretrained weights, outside waveforms or blind-data inputs in training**.

Training-only inspection found repeatable acquisition phase and timing in this dataset.
That makes coherent integration a promising alternative to a larger power-spectrogram
CNN. The notebook evaluates eight scoring alternatives using validation only, selects
one, and evaluates the selected pipeline on all **171,600** validation examples.
An additional audit report excludes every source pulse used for selection.
**No leaderboard score, including 0.995, is guaranteed.**

**Kaggle setup:** attach your existing expanded `radar-ml` dataset and the published
`manifests` dataset. Keep the expanded JSON sidecars. Alternatively attach the smaller
`recorded_dataset` with its sidecars; noise variants are generated on demand using the
supplied expander's algorithm, without writing a 57 GB expansion. Enable a GPU;
one GPU is used. Internet is unnecessary. Also attach the released **participants**
dataset. The default mode trains, freezes, then generates your submission in the
same Kaggle run. Check the paths below and **Run All**.

The run is designed to fit within your **12-hour GPU budget**, and will often finish
much sooner than CNN training. It fits finite template parameters directly, so adding
epochs would not improve this estimator. Larger compute alone does not ensure better
accuracy. Training caches, intermediate validation reports and final bundles are saved.

**Event eligibility:** this is a new experiment, separate from your original frozen
model. The repository requires organizer approval to replace a model after the freeze.
Use this notebook as development work only within the applicable event permissions.

## Measured local reference — 15 October 2026

This exact implementation was fitted on all **52,800 original recorded training
pulses** and evaluated using the published validation identities and the supplied
expander's deterministic noise algorithm. It did not read the participant package.

| Evaluation | Examples | Mean balanced accuracy over all 13 SNRs |
| --- | ---: | ---: |
| Full published validation | 171,600 | **0.99641808** |
| Audit excluding selection source pulses | 137,280 | **0.99646697** |

Selected rule: `coherent_clip:mixture`. Full-validation balanced accuracy at
**−30 dB: 95.3472%**. The old notebook's full-validation mean was 0.89787946.
These are validation results, not blind leaderboard scores. The audit excludes
selection pulses but still uses the same known configurations and capture protocol.

The default Kaggle path points to your **expanded** data and fits templates from
the +30 dB variants, which can give slightly different results. For the closest
reproduction of this reference, attach `recorded_dataset` and set `DATA_ROOT`
accordingly with `DATA_FORMAT="compact"`. Both formats keep the same 80/20 split.

In [ ]:
from pathlib import Path
import copy
import hashlib
import importlib
import json
import shutil
import subprocess
import sys
import time

MODE = "train_and_submit"   # "train_and_submit", "train" (no submission), or "infer"
RUN_NAME = "track2_fourier_bank_v1"
DATA_FORMAT = "auto"         # "auto", "expanded" or "compact"
# Existing mounts from radar-1.ipynb; edit if your Kaggle dataset paths differ.
DATA_ROOT = Path("/kaggle/input/datasets/hadimanes/radar-ml/expanded (2)/expanded")
MANIFEST_PATH = Path("/kaggle/input/datasets/hadimanes/manifests")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
RUN_DIR = WORK_ROOT / RUN_NAME

# To continue a run in another Kaggle session, upload its *_resume.zip as a private
# dataset, extract it, and point here to the directory containing run_settings.json.
RESUME_FROM = None

# Only needed for MODE="infer": extracted frozen directory containing bank.pt.
# In train_and_submit mode, the notebook uses the model it just trained and froze.
FROZEN_BUNDLE = None
# None automatically finds the participant package attached under /kaggle/input.
# If multiple packages are attached, set the directory explicitly, for example:
# BLIND_ROOT = Path("/kaggle/input/your-participants-dataset/participants")
BLIND_ROOT = Path("/kaggle/input/datasets/hadimanes/blind-test/participants")
PREDICTIONS_PATH = WORK_ROOT / "submission_track2.csv"

BATCH_SIZE = 128              # lower to 32 or 64 if GPU memory is limited
WORKERS = 2                  # use 0 if notebook multiprocessing causes problems
assert MODE in {"train_and_submit", "train", "infer"}
TRAIN_MODEL = MODE in {"train", "train_and_submit"}
GENERATE_SUBMISSION = MODE in {"train_and_submit", "infer"}
assert Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in {"", ".", ".."}
if TRAIN_MODEL:
    if RESUME_FROM is not None and not RUN_DIR.exists():
        shutil.copytree(Path(RESUME_FROM), RUN_DIR)
else:
    assert FROZEN_BUNDLE is not None, "Set FROZEN_BUNDLE for inference-only mode"
print("Mode:", MODE, "| run:", RUN_DIR)


## Self-contained source

These cells write the tested runtime and the original data helpers into the working
directory. Nothing needs to be installed from GitHub. The unchanged data helpers keep
FSK upload-name collision recovery, the published manifest checksum, the official
seven labels and the anonymous submission format. Kaggle may rename files containing
`#`; JSON source identities plus array hashes recover the correct pairing.

In [ ]:
%%writefile radarml_track2_runtime.py
"""RadarML Lite Track 2 / Track B: from-scratch STFT classification.
SPDX-License-Identifier: GPL-3.0-or-later
Lookup code adapted from the participant's synapseon.ipynb and the organizer's
GPL-3.0 utilities. RadarML attribution: Ritchie, White and Hosford (2025).
AI coding assistance: OpenAI Codex. No pretrained weights or external data.
"""
import argparse
import csv
import hashlib
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

CLASSES = ("barker", "cw", "fmcw", "fsk", "hyp", "nlfm", "quad")
ALL_SNRS = (30, 25, 20, 15, 10, 5, 0, -5, -10, -15, -20, -25, -30)
SAMPLE_RATE_HZ = 120_000_000
MANIFEST_SHA256 = "4ccc4937f4ba0e9993837bb26a5ba4463f1a880b689a753ad3a0e39b76ebafc7"

# %% File identity and published split
def family(filename):
    parts = Path(filename).stem.lower().split("_")
    if len(parts) < 3:
        raise ValueError(f"Unrecognized waveform filename: {filename}")
    token = parts[1]
    if token.startswith("barker"):
        return "barker"
    if re.fullmatch(r"fsk(?:[#-]?[0-3])?", token):
        return "fsk"
    if token == "fsk" and len(parts) > 2 and parts[2] in {"0", "1", "2", "3"}:
        return "fsk"
    if token in CLASSES:
        return token
    raise ValueError(f"Unrecognized modulation in {filename}: {token}")

def canonical_filename(filename):
    # Kaggle can append " (19)" before an extension. Preserve frequency signs.
    filename = re.sub(r" \(\d+\)(?=\.(?:npy|json)$)", "", filename)
    return re.sub(r"(?<=_)fsk[#_-]?([0-3])(?=_)", r"fsk\1", filename, flags=re.IGNORECASE)

def expanded_name_from_source(source_file):
    """Recreate the organizer's expanded name from JSON's original source_file."""
    source = Path(source_file).name
    if source.endswith("_recorded.npy"):
        return source.replace("_recorded.npy", "_single_channel.npy")
    if source.endswith("_single_channel.npy"):
        return source
    parts = Path(source).stem.split("_")
    if len(parts) < 3:
        raise ValueError(f"Unexpected source_file in JSON sidecar: {source}")
    try:
        parts[-1] = "-".join(f'{240 - int(piece.removesuffix("Mhz"))}Mhz'
                             for piece in parts[-1].split("-"))
    except ValueError as exc:
        raise ValueError(f"Unexpected source frequency in JSON sidecar: {source}") from exc
    return "_".join(parts) + "_single_channel.npy"

def build_file_lookup(root, kind):
    suffix = "_single_channel.npy" if kind == "expanded" else "_recorded.npy"
    files = [p for p in root.rglob("*.npy") if canonical_filename(p.name).endswith(suffix)]
    if not files:
        raise FileNotFoundError(f"No {suffix} files in {root}")
    file_set = set(files)
    arrays, metadata = defaultdict(set), defaultdict(list)
    for path in files:
        arrays[canonical_filename(path.name)].add(path)
    for sidecar in root.rglob("*.json"):
        meta = json.loads(sidecar.read_text(encoding="utf-8"))
        if not isinstance(meta, dict) or not meta.get("source_file"):
            continue
        alias = expanded_name_from_source(meta["source_file"])
        if kind == "compact":
            alias = alias.replace("_single_channel.npy", "_recorded.npy")
        key = canonical_filename(alias)
        metadata[key].append((sidecar, meta))
        paired_array = sidecar.with_suffix(".npy")
        if paired_array in file_set:
            arrays[key].add(paired_array)
    hash_to_paths, path_to_hash = defaultdict(list), {}
    if kind == "expanded":
        fsk_paths = [p for p in files if family(canonical_filename(p.name)) == "fsk"]
        print(f"Hashing {len(fsk_paths)} uploaded FSK arrays to recover their original identities...")
        for count, path in enumerate(fsk_paths, 1):
            digest = lookup_sha256(path)
            hash_to_paths[digest].append(path)
            path_to_hash[path] = digest
            if count % 25 == 0 or count == len(fsk_paths):
                print(f"Hashed {count}/{len(fsk_paths)} FSK arrays")
        for key, entries in metadata.items():
            if "_fsk" in key.lower():
                for _, meta in entries:
                    for path in hash_to_paths.get(meta.get("output_sha256"), []):
                        arrays[key].add(path)
    return {"arrays": arrays, "metadata": metadata,
            "hash_to_paths": hash_to_paths, "path_to_hash": path_to_hash}

def lookup_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def resolve_file(manifest_name, lookup, kind):
    desired = (manifest_name.replace("_recorded.npy", "_single_channel.npy")
               if kind == "expanded" else manifest_name)
    key = canonical_filename(desired)
    if kind == "expanded" and family(manifest_name) == "fsk":
        expected = {meta.get("output_sha256") for _, meta in lookup["metadata"].get(key, [])}
        digests_found = {digest for digest in expected if digest in lookup["hash_to_paths"]}
        if len(digests_found) == 1:
            digest = next(iter(digests_found))
            return sorted(lookup["hash_to_paths"][digest])[0]
        raise ValueError(f"FSK array for {manifest_name} cannot be matched by SHA256. "
                         f"Matching hashes found: {len(digests_found)}. "
                         "The Kaggle dataset may be missing an NPY file or its JSON may be stale.")
    matches = list(lookup["arrays"].get(key, []))
    paired = {sidecar.with_suffix(".npy") for sidecar, _ in lookup["metadata"].get(key, [])}
    if paired:
        matches = [path for path in matches if path in paired]
    if matches:
        shallowest = min(len(path.parts) for path in matches)
        matches = [path for path in matches if len(path.parts) == shallowest]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        expected_hashes = {meta.get("output_sha256") for _, meta in lookup["metadata"].get(key, [])}
        matching = [(path, digest) for path in matches
                    if (digest := lookup_sha256(path)) in expected_hashes]
        if matching and len({digest for _, digest in matching}) == 1:
            return sorted(path for path, _ in matching)[0]
    examples = [p.name for paths in lookup["arrays"].values() for p in paths
                if "fsk" in p.name.lower()][:5]
    raise ValueError(f"Could not uniquely match {manifest_name}: {len(matches)} candidate arrays. "
                     f"Example Kaggle FSK names: {examples}. Check array uploads and JSON checksums.")

def resolve_metadata(manifest_name, path, lookup, kind):
    desired = (manifest_name.replace("_recorded.npy", "_single_channel.npy")
               if kind == "expanded" else manifest_name)
    candidates = lookup["metadata"].get(canonical_filename(desired), [])
    if kind == "expanded" and family(manifest_name) == "fsk":
        digest = lookup["path_to_hash"].get(path)
        matched = [(sidecar, meta) for sidecar, meta in candidates
                   if meta.get("output_sha256") == digest]
        if matched:
            return sorted(matched, key=lambda pair: str(pair[0]))[0]
        raise ValueError(f"FSK metadata checksum missing for {manifest_name}")
    exact = [(sidecar, meta) for sidecar, meta in candidates if sidecar == path.with_suffix(".json")]
    if len(exact) == 1:
        return exact[0]
    if len(candidates) == 1:
        return candidates[0]
    if candidates:
        digest = lookup_sha256(path)
        matched = [(sidecar, meta) for sidecar, meta in candidates
                   if meta.get("output_sha256") == digest]
        if matched:
            return sorted(matched, key=lambda pair: str(pair[0]))[0]
    raise ValueError(f"Could not pair {path.name} with a JSON sidecar for {manifest_name}")

def load_manifest(path):
    path = Path(path)
    if lookup_sha256(path) != MANIFEST_SHA256:
        raise ValueError("Manifest checksum differs from the published release. Use the original CSV, not a regenerated split.")
    with path.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["file", "pulse_index", "split"]:
            raise ValueError("Expected manifest columns: file,pulse_index,split")
        rows = list(reader)
    if not rows:
        raise ValueError("Empty manifest")
    identities = set()
    per_file = Counter()
    for row in rows:
        if row["split"] not in {"train", "validation"}:
            raise ValueError(f"Unexpected split {row['split']}")
        pulse = int(row["pulse_index"])
        if not 0 <= pulse < 100:
            raise ValueError(f"Pulse index outside 0..99: {pulse}")
        identity = (row["file"], pulse)
        if identity in identities:
            raise ValueError(f"Duplicate source pulse: {identity}")
        identities.add(identity)
        per_file[(row["file"], row["split"])] += 1
        row["pulse_index"] = pulse
    for name in {r["file"] for r in rows}:
        if per_file[(name, "train")] != 80 or per_file[(name, "validation")] != 20:
            raise ValueError(f"Manifest is not 80/20 for {name}")
    if len(rows) != 66_000 or len(per_file) != 1_320:
        raise ValueError("Expected the complete published 660-recording, 66,000-pulse manifest")
    print(f"Manifest: {len(rows):,} source pulses across {len(per_file)//2} recordings")
    return rows


# %% Expanded data: no expansion or synthetic-noise generation
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def seed_worker(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed)
    random.seed(seed)


def find_manifest(root, explicit=None):
    if explicit:
        path = Path(explicit)
        if path.is_dir():
            candidates = sorted(path.rglob("split_manifest.csv"))
        else:
            candidates = [path] if path.is_file() else []
    else:
        candidates = [p for p in (Path(root) / "split_manifest.csv",
                                  Path(root).parent / "split_manifest.csv") if p.is_file()]
        if not candidates and Path("/kaggle/input").is_dir():
            candidates = sorted(Path("/kaggle/input").rglob("split_manifest.csv"))
    # Multiple copies are fine only when all have the published identity.
    valid = [p for p in candidates if lookup_sha256(p) == MANIFEST_SHA256]
    if not valid:
        raise FileNotFoundError("Attach the published split_manifest.csv (repository hackathon/ copy); set MANIFEST_PATH to it.")
    return valid[0]


def prepare_development(root, manifest_path):
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"Expanded DATA_ROOT does not exist: {root}. Update the Kaggle path in the settings cell.")
    rows = load_manifest(manifest_path)
    lookup = build_file_lookup(root, "expanded")
    infos, paths_used = {}, set()
    for name in sorted({r["file"] for r in rows}):
        path = resolve_file(name, lookup, "expanded")
        sidecar, meta = resolve_metadata(name, path, lookup, "expanded")
        # Verify that the source identity really maps to this published recording.
        expected_name = name.replace("_recorded.npy", "_single_channel.npy")
        if canonical_filename(expanded_name_from_source(meta["source_file"])) != canonical_filename(expected_name):
            raise ValueError(f"Wrong source identity in {sidecar}")
        snrs = tuple(meta["snr_db"])
        if len(snrs) != 13 or set(snrs) != set(ALL_SNRS):
            raise ValueError(f"Expected all 13 requested SNRs in {sidecar}; found {snrs}")
        if meta.get("source_pulse_indices") != list(range(100)):
            raise ValueError(f"Pulse-axis mapping is not the organizer's 0..99 mapping: {sidecar}")
        if meta.get("sample_rate_hz") != SAMPLE_RATE_HZ or meta.get("channel") != "ADC0":
            raise ValueError(f"Unexpected receiver/sample rate: {sidecar}")
        array = np.load(path, mmap_mode="r", allow_pickle=False)
        if array.shape != (1, 13, 100, 33600) or array.dtype != np.int16:
            raise ValueError(f"Expected expanded int16 (1,13,100,33600): {path}, {array.shape}, {array.dtype}")
        if meta.get("output_shape") != list(array.shape) or meta.get("dtype") != "int16":
            raise ValueError(f"NPY/JSON layout disagreement: {path}")
        del array
        if path in paths_used:
            raise ValueError(f"Two source identities resolve to the same NPY: {path}")
        paths_used.add(path)
        infos[name] = {"path": str(path), "label": CLASSES.index(family(name)),
                       "snr_to_axis": {int(s): i for i, s in enumerate(snrs)}}
    print(f"Validated {len(infos)} expanded arrays; source pulses retain their published split.", flush=True)
    return rows, infos


def build_examples(rows, infos, split, pulse_limit, snrs):
    if split not in {"train", "validation"}:
        raise ValueError("Only the published train and validation splits are used here")
    limit = 80 if split == "train" else 20
    if not 1 <= pulse_limit <= limit or not snrs or not set(snrs).issubset(ALL_SNRS):
        raise ValueError("Invalid source-pulse limit or requested SNR subset")
    grouped = defaultdict(list)
    for row in rows:
        if row["split"] == split:
            grouped[row["file"]].append(row["pulse_index"])
    # SNR first improves sequential validation reads; selection uses original indices.
    return [(name, pulse, int(snr), infos[name]["label"])
            for name in sorted(grouped) for snr in snrs
            for pulse in sorted(grouped[name])[:pulse_limit]]


def interleaved_to_channels(row):
    if row.shape != (33600,) or row.dtype != np.int16:
        raise ValueError(f"Expected one interleaved int16 pulse: {row.shape}, {row.dtype}")
    return np.stack((row[0::2], row[1::2])).astype(np.float32) / 32768.0


class DevelopmentDataset(Dataset):
    def __init__(self, examples, infos):
        self.examples, self.infos = examples, infos

    @staticmethod
    @lru_cache(maxsize=24)
    def _array(path):
        return np.load(path, mmap_mode="r", allow_pickle=False)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        name, pulse, snr, label = self.examples[index]
        info = self.infos[name]
        row = self._array(info["path"])[0, info["snr_to_axis"][snr], pulse]
        # Label and requested SNR are returned for supervision/metrics only.
        return torch.from_numpy(interleaved_to_channels(row)), label, snr


def make_loader(dataset, batch_size, workers, seed, sampler=None):
    options = dict(batch_size=batch_size, num_workers=workers,
                   pin_memory=torch.cuda.is_available(), shuffle=False,
                   generator=torch.Generator().manual_seed(seed), worker_init_fn=seed_worker)
    if workers:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(dataset, sampler=sampler, **options)


# %% Track B representation and model, initialized from scratch
class TwoResolutionSTFT(nn.Module):
    def __init__(self, n_ffts=(256, 1024), hop_length=64, image_size=(256, 256)):
        super().__init__()
        self.n_ffts, self.hop_length, self.image_size = tuple(n_ffts), hop_length, tuple(image_size)
        for n_fft in self.n_ffts:
            self.register_buffer(f"window_{n_fft}", torch.hann_window(n_fft), persistent=False)

    def forward(self, x):
        # FFT remains float32/complex64 even during mixed-precision training.
        with torch.autocast(device_type=x.device.type, enabled=False):
            x = x.float()
            rms = x.square().sum(1).mean(1, keepdim=True).sqrt().clamp_min(1e-6)
            iq = torch.complex(x[:, 0], x[:, 1]) / rms
            views = []
            for n_fft in self.n_ffts:
                window = getattr(self, f"window_{n_fft}")
                z = torch.stft(iq, n_fft=n_fft, hop_length=self.hop_length,
                               win_length=n_fft, window=window, center=True,
                               pad_mode="constant", onesided=False, return_complex=True)
                power = torch.fft.fftshift(z.abs().square(), dim=1) / window.square().sum()
                # Fixed acquisition window, never filename pulse duration or SNR label.
                # Use only frames fully inside the 0..20 us noise interval.
                first = math.ceil((n_fft / 2) / self.hop_length)
                stop = math.floor((2400 - n_fft / 2) / self.hop_length) + 1
                noise_floor = power[:, :, first:stop].mean((1, 2), keepdim=True).clamp_min(1e-8)
                view = torch.log1p(power / noise_floor).unsqueeze(1)
                # Averaging preserves narrow-band energy without taking only the largest bin.
                views.append(F.adaptive_avg_pool2d(view, self.image_size))
            return torch.cat(views, dim=1)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False),
            nn.GroupNorm(8, out_channels), nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.GroupNorm(8, out_channels))
        self.skip = (nn.Identity() if in_channels == out_channels and stride == 1 else
                     nn.Conv2d(in_channels, out_channels, 1, stride, bias=False))

    def forward(self, x):
        return F.silu(self.main(x) + self.skip(x))


class Track2Model(nn.Module):
    def __init__(self, n_ffts=(256, 1024), hop_length=64, image_size=(256, 256), dropout=0.2):
        super().__init__()
        self.representation = TwoResolutionSTFT(n_ffts, hop_length, image_size)
        self.features = nn.Sequential(
            nn.Conv2d(len(n_ffts), 24, 5, 2, 2, bias=False),
            nn.GroupNorm(8, 24), nn.SiLU(inplace=True),
            ResidualBlock(24, 48, 2), ResidualBlock(48, 48),
            ResidualBlock(48, 96, 2), ResidualBlock(96, 96),
            ResidualBlock(96, 128, 2), ResidualBlock(128, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
                                  nn.Dropout(dropout), nn.Linear(128 * 4 * 4, 256),
                                  nn.SiLU(), nn.Dropout(dropout), nn.Linear(256, len(CLASSES)))

    def forward(self, iq):
        return self.head(self.features(self.representation(iq)))


# %% Organizer-compatible metric and measured evaluation
def report_from_matrices(matrices):
    per_snr = []
    for snr in ALL_SNRS:
        if snr not in matrices:
            continue
        matrix = matrices[snr]
        totals = matrix.sum(1)
        if (totals == 0).any():
            raise ValueError(f"Requested SNR {snr} lacks one or more classes")
        per_snr.append({"requested_snr_db": int(snr),
                        "balanced_accuracy": float(np.mean(matrix.diagonal() / totals)),
                        "accuracy": float(matrix.trace() / matrix.sum()),
                        "class_recall": (matrix.diagonal() / totals).tolist(),
                        "confusion_matrix": matrix.tolist()})
    low = [p["balanced_accuracy"] for p in per_snr if p["requested_snr_db"] in (-20, -25, -30)]
    return {"classes": list(CLASSES), "samples": int(sum(m.sum() for m in matrices.values())),
            "mean_balanced_accuracy": float(np.mean([p["balanced_accuracy"] for p in per_snr])),
            "lowest_three_mean_balanced_accuracy": float(np.mean(low)) if len(low) == 3 else None,
            "per_snr": per_snr}


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


@torch.inference_mode()
def evaluate(model, loader, device, snrs=ALL_SNRS):
    model.eval()
    matrices = {int(s): np.zeros((7, 7), dtype=np.int64) for s in snrs}
    synchronize(device)
    start = time.perf_counter()
    for signals, labels, batch_snrs in loader:
        prediction = model(signals.to(device, non_blocking=True)).argmax(1).cpu().numpy()
        labels, batch_snrs = labels.numpy(), batch_snrs.numpy()
        for snr in snrs:
            mask = batch_snrs == snr
            np.add.at(matrices[int(snr)], (labels[mask], prediction[mask]), 1)
    synchronize(device)
    seconds = time.perf_counter() - start
    report = report_from_matrices(matrices)
    report.update(inference_seconds=seconds, seconds_per_pulse=seconds / report["samples"],
                  timing_scope="DataLoader, disk reads, transfer, RMS/noise normalization, both STFTs, CNN, metric accumulation; FP32")
    return report


def hardware_info(device):
    return {"device": str(device), "cpu": platform.processor(), "platform": platform.platform(),
            "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
            "gpu_memory_bytes": torch.cuda.get_device_properties(device).total_memory if device.type == "cuda" else None,
            "python": sys.version, "numpy": str(np.__version__), "torch": str(torch.__version__),
            "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version()}


def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


# %% Training, checkpoint selection and reproducible freeze bundle
DEFAULT_CONFIG = {
    "seed": 42, "train_pulses_per_file": 80, "train_snrs": list(ALL_SNRS),
    "selection_pulses_per_file": 4, "batch_size": 64, "workers": 2,
    "epochs": 20, "samples_per_epoch": 105600, "patience": 6,
    "learning_rate": 0.0006, "weight_decay": 0.01, "label_smoothing": 0.02,
    "train_amp": True,
    "model_kwargs": {"n_ffts": [256, 1024], "hop_length": 64,
                     "image_size": [256, 256], "dropout": 0.2},
}


def train_and_freeze(data_root, manifest_path, output, config=None, prepared=None):
    cfg = json.loads(json.dumps(DEFAULT_CONFIG if config is None else config))
    output = Path(output)
    # Never overwrite frozen artifacts during a rerun or a new experiment.
    if output.exists() and any(output.iterdir()):
        raise FileExistsError(f"Output is not empty: {output}. Choose a new RUN_NAME for a new experiment.")
    output.mkdir(parents=True, exist_ok=True)
    if not 1 <= cfg["selection_pulses_per_file"] <= 20:
        raise ValueError("Selection validation uses 1..20 published validation pulses per recording")
    if cfg["epochs"] < 1 or cfg["samples_per_epoch"] < 7:
        raise ValueError("Need at least one epoch and seven samples per epoch")
    seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    setup_start = time.perf_counter()
    manifest_path = find_manifest(data_root, manifest_path)
    if prepared is None:
        rows, infos = prepare_development(data_root, manifest_path)
        preparation_seconds = 0.0
    else:
        rows, infos, preparation_seconds = prepared
    train_ex = build_examples(rows, infos, "train", cfg["train_pulses_per_file"], cfg["train_snrs"])
    select_ex = build_examples(rows, infos, "validation", cfg["selection_pulses_per_file"], ALL_SNRS)
    full_ex = build_examples(rows, infos, "validation", 20, ALL_SNRS)
    train_ids = {(e[0], e[1]) for e in train_ex}
    assert not train_ids.intersection((e[0], e[1]) for e in full_ex), "Source-pulse leakage"
    labels = np.array([e[3] for e in train_ex], dtype=np.int64)
    counts = np.bincount(labels, minlength=7)
    if (counts == 0).any():
        raise ValueError("All seven classes must be present in training")
    sampler = WeightedRandomSampler(torch.as_tensor(1.0 / counts[labels], dtype=torch.double),
                                    cfg["samples_per_epoch"], replacement=True,
                                    generator=torch.Generator().manual_seed(cfg["seed"]))
    train_loader = make_loader(DevelopmentDataset(train_ex, infos), cfg["batch_size"], cfg["workers"], cfg["seed"], sampler)
    select_loader = make_loader(DevelopmentDataset(select_ex, infos), cfg["batch_size"], cfg["workers"], cfg["seed"])
    print("Eligible training examples:", len(train_ex), "| sampled per epoch:", cfg["samples_per_epoch"], flush=True)
    print("Eligible training class counts:", dict(zip(CLASSES, counts.tolist())), flush=True)
    print("Selection validation:", len(select_ex), "| final full validation:", len(full_ex), flush=True)
    model = Track2Model(**cfg["model_kwargs"]).to(device)
    parameter_count = sum(p.numel() for p in model.parameters())
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    use_amp = bool(cfg["train_amp"] and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])
    best_score, best_low, best_epoch, stale = -1.0, -1.0, 0, 0
    history, fit_seconds, selection_seconds = [], 0.0, 0.0
    setup_seconds = time.perf_counter() - setup_start + preparation_seconds
    synchronize(device)
    start = time.perf_counter()
    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        epoch_start = time.perf_counter()
        total_loss, seen = 0.0, 0
        for step, (signals, targets, _) in enumerate(train_loader, 1):
            signals, targets = signals.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(signals), targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach()) * len(targets)
            seen += len(targets)
            if step == 1 or step % 100 == 0:
                print(f"Epoch {epoch}/{cfg['epochs']}, batch {step}/{len(train_loader)}, loss {total_loss/seen:.4f}", flush=True)
        synchronize(device)
        fit_seconds += time.perf_counter() - epoch_start
        report = evaluate(model, select_loader, device)
        selection_seconds += report["inference_seconds"]
        score, low = report["mean_balanced_accuracy"], report["lowest_three_mean_balanced_accuracy"]
        record = {"epoch": epoch, "train_loss": total_loss / seen,
                  "learning_rate": optimizer.param_groups[0]["lr"],
                  "selection_mean_balanced_accuracy": score, "selection_lowest_three": low}
        history.append(record)
        print(json.dumps(record), flush=True)
        better = score > best_score + 1e-6 or (abs(score - best_score) <= 1e-6 and low > best_low + 1e-6)
        if better:
            best_score, best_low, best_epoch, stale = score, low, epoch, 0
            torch.save({"state_dict": model.state_dict(), "classes": list(CLASSES),
                        "track": "B", "model_kwargs": cfg["model_kwargs"]}, output / "model.pt")
            report.update(selected_epoch=epoch, pulses_per_recording=cfg["selection_pulses_per_file"],
                          full_published_validation=cfg["selection_pulses_per_file"] == 20)
            write_json(output / "selection_validation.json", report)
        else:
            stale += 1
        write_json(output / "training_history.json", history)
        scheduler.step()
        if stale >= cfg["patience"]:
            print("Early stopping on validation only.", flush=True)
            break
    synchronize(device)
    development_seconds = time.perf_counter() - start
    # Evaluate exactly the saved best checkpoint, never the final in-memory epoch.
    checkpoint = torch.load(output / "model.pt", map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["state_dict"])
    del checkpoint, optimizer, train_loader, select_loader
    full_loader = make_loader(DevelopmentDataset(full_ex, infos), cfg["batch_size"], cfg["workers"], cfg["seed"])
    full_report = evaluate(model, full_loader, device)
    assert full_report["samples"] == 171600
    full_report.update(selected_epoch=best_epoch, pulses_per_recording=20, full_published_validation=True)
    write_json(output / "validation_report.json", full_report)
    cfg.update(track="B", track_number=2, classes=list(CLASSES), sample_rate_hz=SAMPLE_RATE_HZ,
               manifest_sha256=MANIFEST_SHA256, train_source_pulses=len(train_ids),
               eligible_training_examples=len(train_ex), eligible_class_counts=dict(zip(CLASSES, counts.tolist())),
               subset_rule="First N ascending published pulse indices per recording, separately within each split",
               sampler="Inverse-class-count weighted replacement sampling; every selected SNR has equal sampling probability within each class",
               optimizer="AdamW", schedule="CosineAnnealingLR by epoch", gradient_clip_norm=5.0,
               selection_rule="Mean balanced accuracy across all 13 SNRs on selection validation; tolerance 1e-6, lowest-three-SNR tie-break",
               selected_epoch=best_epoch, epochs_completed=len(history), training_draws=len(history)*cfg["samples_per_epoch"],
               inference_precision="float32", inference_augmentation="none", training_augmentation="none beyond supplied SNR variants",
               model_parameters=parameter_count, model_bytes=(output / "model.pt").stat().st_size,
               setup_seconds=setup_seconds, fit_seconds=fit_seconds, selection_validation_seconds=selection_seconds,
               development_seconds=development_seconds, hardware=hardware_info(device),
               preprocessing="Complex RMS per pulse; centered two-sided Hann STFT with zero padding; log1p(power / per-pulse noise-window mean power); frequency shift; adaptive mean pooling",
               pretrained=False, external_training_data=False)
    write_json(output / "config.json", cfg)
    shutil.copyfile(Path(__file__), output / "radarml_track2_runtime.py")
    shutil.copyfile(manifest_path, output / "split_manifest.csv")
    # Public version pins accept the CUDA build already installed on Kaggle.
    # Exact build, CUDA/cuDNN and hardware are retained in config.json.
    torch_public_version = str(torch.__version__).split("+", 1)[0]
    (output / "requirements.txt").write_text(
        f"# Exact CUDA/build details: config.json; use the original Kaggle environment.\n"
        f"numpy=={np.__version__}\ntorch=={torch_public_version}\n", encoding="utf-8")
    (output / "inference_command.txt").write_text(
        "python radarml_track2_runtime.py infer --bundle . --blind-root /path/to/blind_test_participants --output /path/to/predictions.csv\n", encoding="utf-8")
    summary = f"""RadarML Lite — Track 2 / B (time-frequency)

Method: two-sided Hann STFT at {cfg['model_kwargs']['n_ffts']} samples, hop
{cfg['model_kwargs']['hop_length']}; per-pulse RMS/noise-window normalization,
log power and adaptive pooling; a residual 2D CNN with GroupNorm, trained
from scratch with class-balanced replacement sampling, AdamW and cosine LR.
No filename, ID, metadata, pulse index or requested-SNR label is a model input.

Published split: {MANIFEST_SHA256}. Source-pulse training subset:
{len(train_ids):,} ({cfg['train_pulses_per_file']} / 80 per recording), SNRs
{cfg['train_snrs']}. All variants retain the source pulse's split. Seed:
{cfg['seed']}. Draws per epoch: {cfg['samples_per_epoch']:,}; this sampling
budget does not guarantee every eligible variant is visited each epoch.
Training draws actually used: {cfg['training_draws']:,}.

Checkpoint selected on {cfg['selection_pulses_per_file']} validation pulses
per recording at all 13 SNRs; selected epoch: {best_epoch}. After selection,
the saved checkpoint was evaluated on all 171,600 published validation
examples: mean balanced accuracy {full_report['mean_balanced_accuracy']:.6f};
lowest-three-SNR mean {full_report['lowest_three_mean_balanced_accuracy']:.6f}.
See validation_report.json for per-SNR scores and all confusion matrices.
Validation pulses were never fitted or added to final training.

Hardware/dependency versions: config.json and requirements.txt. Model:
{parameter_count:,} parameters, {cfg['model_bytes']:,} bytes. Fit-only time:
{fit_seconds:.1f} s; fit plus selection/checkpoint overhead:
{development_seconds:.1f} s; dataset setup: {setup_seconds:.1f} s.
Full-validation inference: {full_report['seconds_per_pulse']:.6f} s/pulse,
including loading, transfers, normalization, STFTs and CNN. Blind-run timing
is recorded separately in predictions.csv.inference.json after official release.

Limitations: validation and blind test use known waveform configurations;
results do not establish generalization to other receivers or configurations.
Requested SNR differs from achieved SNR, and int16 clipping affects low SNRs.
Pooling trades spectral/time detail for speed. No blind score is known here.

Attribution: Ritchie, White and Hosford (2025), RadarML v1,
https://doi.org/10.5522/04/30752767.v1; organizer Radar_ML_lite code and rules,
GPL-3.0; file-identity lookup adapted from the participant's synapseon.ipynb.
AI assistance: OpenAI Codex helped inspect documentation and implement/debug
the notebook and runtime. No external learned weights or training data used.

Before submitting: identify your team/name and organizer-issued pipeline
identifier in the submission message; keep the hashed bundle unchanged.
Send this frozen bundle at the announced freeze deadline; after blind release
run the inference command, then add predictions.csv and its inference report.
"""
    (output / "method_summary.txt").write_text(summary, encoding="utf-8")
    hashes = {p.name: lookup_sha256(p) for p in sorted(output.iterdir()) if p.is_file()}
    write_json(output / "frozen.sha256.json", hashes)
    print("Full validation mean balanced accuracy:", full_report["mean_balanced_accuracy"], flush=True)
    print("Frozen artifacts:", output, flush=True)
    return full_report


# %% Blind inference: frozen code/config/weights, anonymous shard loader
def verify_bundle(bundle):
    bundle = Path(bundle)
    hashes = json.loads((bundle / "frozen.sha256.json").read_text(encoding="utf-8"))
    required = {"model.pt", "config.json", "radarml_track2_runtime.py", "requirements.txt"}
    if not required.issubset(hashes):
        raise ValueError("Incomplete freeze manifest")
    for name, expected in hashes.items():
        if Path(name).name != name or lookup_sha256(bundle / name) != expected:
            raise ValueError(f"Frozen artifact changed or invalid: {name}")
    # The function actually executing must also be the frozen runtime.
    if lookup_sha256(Path(__file__)) != hashes["radarml_track2_runtime.py"]:
        raise ValueError("Execute the runtime inside the frozen bundle")
    return hashes


def read_blind_index(blind_root, expected_count=171600):
    root = Path(blind_root).resolve()
    if not (root / "index.csv").is_file() and (root / "test" / "index.csv").is_file():
        root = root / "test"
    with (root / "index.csv").open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if not {"sample_id", "file", "row"}.issubset(reader.fieldnames or []):
            raise ValueError("Anonymous index must provide sample_id,file,row")
        rows = list(reader)
    template = root / "submission_template.csv"
    if not template.is_file():
        template = root.parent / "submission_template.csv"
    with template.open(newline="", encoding="utf-8-sig") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["sample_id", "predicted_label"]:
            raise ValueError("Unexpected submission template columns")
        ids = [r["sample_id"] for r in reader]
    index_ids = [r["sample_id"] for r in rows]
    if (len(rows) != expected_count or any(not i for i in index_ids) or
            len(set(index_ids)) != len(index_ids) or len(set(ids)) != len(ids) or set(index_ids) != set(ids)):
        raise ValueError("Blind index/template must contain every expected anonymous ID once")
    locations = set()
    for row in rows:
        relative = Path(row["file"])
        path = (root / relative).resolve()
        if relative.is_absolute() or not path.is_relative_to(root):
            raise ValueError("Shard path escapes the blind directory")
        row["row"] = int(row["row"])
        if row["row"] < 0 or (row["file"], row["row"]) in locations:
            raise ValueError("Duplicate or invalid blind shard row")
        locations.add((row["file"], row["row"]))
    return root, rows, ids


class BlindDataset(Dataset):
    def __init__(self, root, rows):
        self.root, self.rows = Path(root), rows

    @staticmethod
    @lru_cache(maxsize=8)
    def _array(path):
        array = np.load(path, mmap_mode="r", allow_pickle=False)
        if array.ndim != 2 or array.shape[1] != 33600 or array.dtype != np.int16:
            raise ValueError(f"Expected int16 blind shard (N,33600): {path}")
        return array

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        info = self.rows[index]
        row = self._array(str(self.root / info["file"]))[info["row"]]
        return torch.from_numpy(interleaved_to_channels(row))


def predict_blind(bundle, blind_root, output, batch_size=64, workers=2):
    bundle, output = Path(bundle), Path(output)
    hashes = verify_bundle(bundle)
    cfg = json.loads((bundle / "config.json").read_text(encoding="utf-8"))
    if cfg["track"] != "B" or cfg["classes"] != list(CLASSES) or cfg["inference_precision"] != "float32":
        raise ValueError("Unexpected frozen pipeline settings")
    seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(bundle / "model.pt", map_location="cpu", weights_only=True)
    if (checkpoint["track"] != "B" or checkpoint["classes"] != list(CLASSES) or
            checkpoint["model_kwargs"] != cfg["model_kwargs"]):
        raise ValueError("Checkpoint disagrees with frozen configuration")
    model = Track2Model(**cfg["model_kwargs"]).to(device)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    del checkpoint
    root, rows, template_ids = read_blind_index(blind_root)
    # Storage-only ordering; IDs are used solely to put outputs into the template.
    rows.sort(key=lambda r: (r["file"], r["row"]))
    loader = make_loader(BlindDataset(root, rows), batch_size, workers, cfg["seed"])
    synchronize(device)
    start = time.perf_counter()
    predictions = []
    with torch.inference_mode():
        for step, signals in enumerate(loader, 1):
            predictions.extend(model(signals.to(device, non_blocking=True)).argmax(1).cpu().tolist())
            if step == 1 or step % 100 == 0:
                print(f"Blind inference: {len(predictions):,}/{len(rows):,}", flush=True)
    synchronize(device)
    elapsed = time.perf_counter() - start
    if len(predictions) != len(rows):
        raise RuntimeError("Incomplete blind inference")
    by_id = {r["sample_id"]: CLASSES[p] for r, p in zip(rows, predictions)}
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_name(output.name + ".partial")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(("sample_id", "predicted_label"))
        writer.writerows((sid, by_id[sid]) for sid in template_ids)
    # Check the actual serialized file before publishing it as predictions.csv.
    with temporary.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["sample_id", "predicted_label"]
        saved = list(reader)
    assert [r["sample_id"] for r in saved] == template_ids
    assert all(r["predicted_label"] in CLASSES for r in saved)
    temporary.replace(output)
    report = {"samples": len(rows), "inference_seconds": elapsed, "seconds_per_pulse": elapsed / len(rows),
              "timing_scope": "Cold DataLoader startup, shard loading, transfers, normalization, both STFTs, CNN and prediction collection; excludes model/index loading and CSV writing",
              "precision": "float32", "hardware": hardware_info(device),
              "frozen_artifact_hashes": hashes, "predictions_sha256": lookup_sha256(output),
              "batch_size": batch_size, "workers": workers}
    write_json(output.with_name(output.name + ".inference.json"), report)
    print("Predictions:", output, "| seconds/pulse:", report["seconds_per_pulse"], flush=True)
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    infer = commands.add_parser("infer", help="Inference after official blind-test release only")
    infer.add_argument("--bundle", required=True)
    infer.add_argument("--blind-root", required=True)
    infer.add_argument("--output", required=True)
    infer.add_argument("--batch-size", type=int, default=64)
    infer.add_argument("--workers", type=int, default=2)
    args = parser.parse_args()
    predict_blind(args.bundle, args.blind_root, args.output, args.batch_size, args.workers)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile radarml_track2_bank.py
"""RadarML Lite Track 2: learned complex Fourier templates.
SPDX-License-Identifier: GPL-3.0-or-later
Uses training pulses only; no pretrained weights or external waveforms.
AI assistance: OpenAI Codex. Data helpers: radarml_track2_runtime.py.
"""
import argparse
from collections import Counter, defaultdict
import csv
import hashlib
import json
from functools import lru_cache
from pathlib import Path
import shutil
import sys
import time

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
import radarml_track2_runtime as data

CLASSES, ALL_SNRS = data.CLASSES, data.ALL_SNRS
CANDIDATES = tuple(f"{mode}:{pool}" for mode in
                   ("coherent", "coherent_clip", "unknown_phase", "free_amplitude")
                   for pool in ("max", "mixture"))
DEFAULT_CONFIG = dict(seed=42, train_pulses_per_file=80, template_snr=30,
                      selection_pulses_per_file=4, batch_size=128, workers=2,
                      candidates=list(CANDIDATES), window=[2400, 14640])


def prepare_development(root, manifest, kind="auto"):
    root = Path(root)
    if kind == "auto":
        compact = any(data.canonical_filename(p.name).endswith("_recorded.npy")
                      for p in root.rglob("*.npy"))
        expanded = any(data.canonical_filename(p.name).endswith("_single_channel.npy")
                       for p in root.rglob("*.npy"))
        if compact == expanded:
            raise ValueError("Choose a DATA_ROOT with exactly one development format, or set DATA_FORMAT")
        kind = "compact" if compact else "expanded"
    if kind == "expanded":
        rows, infos = data.prepare_development(root, manifest)
        for name, info in infos.items():
            info.update(kind=kind)
        return rows, infos
    if kind != "compact":
        raise ValueError("DATA_FORMAT must be auto, compact or expanded")
    rows = data.load_manifest(manifest)
    lookup = data.build_file_lookup(root, "compact")
    infos, used = {}, set()
    for i, name in enumerate(sorted({r["file"] for r in rows}), 1):
        path = data.resolve_file(name, lookup, "compact")
        sidecar, meta = data.resolve_metadata(name, path, lookup, "compact")
        if data.canonical_filename(data.expanded_name_from_source(meta["source_file"])) != data.canonical_filename(name.replace("_recorded", "_single_channel")):
            raise ValueError(f"Source identity mismatch: {sidecar}")
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        if (arr.shape != (1, 100, 33600) or arr.dtype != np.int16 or
                meta.get("format") != "radarml-recorded-v1" or
                meta.get("source_pulse_indices") != list(range(100)) or
                meta.get("channel") != "ADC0" or meta.get("sample_rate_hz") != 120_000_000 or
                meta.get("output_shape") != list(arr.shape) or meta.get("dtype") != "int16"):
            raise ValueError(f"Invalid compact layout or metadata: {path}")
        if path in used or data.lookup_sha256(path) != meta.get("output_sha256"):
            raise ValueError(f"Duplicate or corrupted compact recording: {path}")
        used.add(path)
        infos[name] = dict(path=str(path), kind=kind, source_file=meta["source_file"],
                           label=CLASSES.index(data.family(name)), snr_to_axis={})
        if i % 100 == 0 or i == 660:
            print(f"Verified {i}/660 compact recordings", flush=True)
    return rows, infos


def add_supplied_noise(raw, snr, source_file, pulse, seed=42):
    """Same generator, power windows, seed convention and clipping as the ZIP's expander."""
    z = raw[::2].astype(np.float64) + 1j * raw[1::2].astype(np.float64)
    noise = np.mean(np.abs(z[:2400]) ** 2)
    signal = np.mean(np.abs(z[2520:14280]) ** 2) - noise
    if not np.isfinite(signal) or signal <= 0:
        raise ValueError("Nonpositive source signal power")
    sigma = np.sqrt(max(0., signal / (10. ** (snr / 10.)) - noise) / 2.)
    key = f"{seed}_{source_file}_{snr}_{pulse}"
    rng = np.random.default_rng(int(hashlib.md5(key.encode()).hexdigest()[:8], 16))
    z = z + rng.normal(0, sigma, z.size) + 1j * rng.normal(0, sigma, z.size)
    out = np.empty(raw.shape, dtype=np.float64)
    out[::2], out[1::2] = z.real, z.imag
    return np.clip(out, -32768, 32767).astype(np.int16)


@lru_cache(maxsize=8)
def mapped_array(path):
    return np.load(path, mmap_mode="r", allow_pickle=False)


class DevelopmentDataset(Dataset):
    def __init__(self, examples, infos, selection_ids=(), noise_seed=42):
        self.examples, self.infos = examples, infos
        self.selection_ids, self.noise_seed = set(selection_ids), noise_seed

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        name, pulse, snr, label = self.examples[index]
        info = self.infos[name]
        arr = mapped_array(info["path"])
        if info["kind"] == "expanded":
            raw = arr[0, info["snr_to_axis"][snr], pulse]
        else:
            raw = add_supplied_noise(arr[0, pulse], snr, info["source_file"], pulse, self.noise_seed)
        return (torch.from_numpy(data.interleaved_to_channels(raw)), label, snr,
                (name, pulse) in self.selection_ids)


def selected_identities(rows, count):
    grouped = defaultdict(list)
    for r in rows:
        if r["split"] == "validation":
            grouped[r["file"]].append(r["pulse_index"])
    return {(name, idx) for name, indices in grouped.items() for idx in sorted(indices)[:count]}


def fit_bank(rows, infos, output, cfg):
    """MLE waveform estimates, with optional phase alignment derived from training only."""
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    grouped = defaultdict(list)
    for r in rows:
        if r["split"] == "train":
            grouped[r["file"]].append(r["pulse_index"])
    if set(grouped) != set(infos):
        raise ValueError("Training recordings disagree with data lookup")
    templates, labels, coherence, fit_seconds = [], [], [], 0.
    start = time.perf_counter()
    for i, name in enumerate(sorted(grouped)):
        indices = sorted(grouped[name])[:cfg["train_pulses_per_file"]]
        info = infos[name]
        cache = output / f"template_{i:04d}.pt"
        identity = dict(file=name, indices=indices, template_snr=cfg["template_snr"],
                        window=cfg["window"], label=info["label"])
        if cache.exists():
            item = torch.load(cache, map_location="cpu", weights_only=True)
            if item["identity"] != identity:
                raise ValueError("Resume cache settings changed; use a new RUN_NAME")
        else:
            tick = time.perf_counter()
            arr = mapped_array(info["path"])
            if info["kind"] == "compact":
                raw = arr[0, indices]
            else:
                raw = arr[0, info["snr_to_axis"][cfg["template_snr"]], indices]
            z = (raw[:, ::2].astype(np.float32) + 1j * raw[:, 1::2].astype(np.float32)) / 32768.
            left, right = cfg["window"]
            z[:, :left], z[:, right:] = 0, 0
            ref = z[0]
            angles = np.angle(z @ ref.conj())
            lock = float(abs(np.exp(1j * angles).mean()))
            # Preserve observed phase whenever stable. When unstable, align training
            # pulses; validation can select the phase-invariant likelihood instead.
            mean = z.mean(0) if lock >= 0.98 else (z * np.exp(-1j * angles)[:, None]).mean(0)
            item = dict(identity=identity, template=torch.from_numpy(mean.astype(np.complex64)),
                        phase_lock=lock, fit_seconds=time.perf_counter() - tick)
            temporary = cache.with_suffix(".partial")
            torch.save(item, temporary)
            temporary.replace(cache)
        templates.append(item["template"])
        labels.append(info["label"])
        coherence.append(item["phase_lock"])
        fit_seconds += item["fit_seconds"]
        if i % 50 == 0 or i + 1 == len(grouped):
            print(f"Training templates: {i + 1}/{len(grouped)}", flush=True)
    bank = dict(templates=torch.stack(templates), labels=torch.tensor(labels),
                classes=list(CLASSES), track="B", model_family="complex_fft_template_bank")
    report = dict(source_pulses=len(grouped) * cfg["train_pulses_per_file"],
                  templates=len(grouped), fit_seconds=fit_seconds,
                  current_session_fit_seconds=time.perf_counter() - start,
                  min_phase_lock=min(coherence), median_phase_lock=float(np.median(coherence)),
                  phase_lock=coherence,
                  training_snr="Original recorded pulses" if next(iter(infos.values()))["kind"] == "compact"
                               else cfg["template_snr"],
                  fit_method="Mean waveform; training-derived phase alignment only if phase lock < 0.98")
    return bank, report


class FourierBank(nn.Module):
    def __init__(self, checkpoint):
        super().__init__()
        if (checkpoint["classes"] != list(CLASSES) or checkpoint["track"] != "B" or
                checkpoint["model_family"] != "complex_fft_template_bank"):
            raise ValueError("Unexpected bank format")
        templates, labels = checkpoint["templates"], checkpoint["labels"]
        if (templates.ndim != 2 or templates.shape[1] != 16800 or
                templates.dtype != torch.complex64 or labels.shape != templates.shape[:1] or
                set(labels.tolist()) != set(range(7)) or not torch.isfinite(templates).all()):
            raise ValueError("Invalid template bank")
        energy = templates.abs().square().sum(1)
        if (energy <= 0).any():
            raise ValueError("Empty training template")
        self.register_buffer("kernels", torch.fft.fft(templates, norm="ortho").conj().T.contiguous())
        self.register_buffer("energy", energy)
        self.register_buffer("labels", labels)
        self.groups = [torch.where(labels == c)[0].tolist() for c in range(7)]

    def components(self, signals):
        # Full complex Fourier coefficients retain coherent information discarded
        # by log-power spectrograms. Parseval gives the template inner product.
        z = torch.complex(signals[:, 0].float(), signals[:, 1].float())
        dot = torch.fft.fft(z, norm="ortho") @ self.kernels
        noise = z[:, :2400].abs().square().mean(1, keepdim=True).clamp_min(1e-10)
        clipped = ((signals <= -1.) | (signals >= 32767. / 32768.)).float().mean((1, 2))[:, None]
        alpha = (1. - clipped).clamp(0.1, 1.)
        return dot, noise, alpha

    def scores(self, signals, candidates):
        dot, noise, alpha = self.components(signals)
        modes = {c.split(":")[0] for c in candidates}
        raw = {}
        if "coherent" in modes:
            raw["coherent"] = (2. * dot.real - self.energy) / noise
        if "coherent_clip" in modes:
            raw["coherent_clip"] = (2. * alpha * dot.real - alpha.square() * self.energy) / noise
        if "unknown_phase" in modes:
            magnitude = 2. * dot.abs() / noise
            # Stable phase-marginalized known-amplitude Gaussian likelihood.
            raw["unknown_phase"] = (magnitude + torch.special.i0e(magnitude).clamp_min(1e-30).log()
                                     - self.energy / noise)
        if "free_amplitude" in modes:
            raw["free_amplitude"] = dot.abs().square() / (self.energy * noise)
        result = {}
        for candidate in candidates:
            mode, pool = candidate.split(":")
            if mode not in raw or pool not in {"max", "mixture"}:
                raise ValueError(f"Unknown candidate: {candidate}")
            values = raw[mode]
            if pool == "max":
                cls = [values[:, idx].amax(1) for idx in self.groups]
            else:
                # Uniform class prior and uniform template prior WITHIN a class.
                # This prevents classes with more configurations gaining extra mass.
                cls = [torch.logsumexp(values[:, idx], 1) - np.log(len(idx)) for idx in self.groups]
            result[candidate] = torch.stack(cls, 1)
        return result

    def forward(self, signals, candidate="coherent_clip:mixture"):
        return self.scores(signals, [candidate])[candidate]


@torch.inference_mode()
def evaluate_candidates(model, loader, device, candidates, snrs=ALL_SNRS, collect_audit=False):
    model.eval()
    matrices = {c: {s: np.zeros((7, 7), dtype=np.int64) for s in snrs} for c in candidates}
    audit = {c: {s: np.zeros((7, 7), dtype=np.int64) for s in snrs} for c in candidates}
    data.synchronize(device)
    start, seen = time.perf_counter(), 0
    for step, (signals, labels, batch_snrs, in_selection) in enumerate(loader, 1):
        scores = model.scores(signals.to(device, non_blocking=True), candidates)
        labels, batch_snrs = labels.numpy(), batch_snrs.numpy()
        for c, logits in scores.items():
            prediction = logits.argmax(1).cpu().numpy()
            for s in snrs:
                mask = batch_snrs == s
                np.add.at(matrices[c][s], (labels[mask], prediction[mask]), 1)
                if collect_audit:
                    holdout = mask & ~in_selection.numpy()
                    np.add.at(audit[c][s], (labels[holdout], prediction[holdout]), 1)
        seen += len(signals)
        if step == 1 or step % 100 == 0:
            print(f"Validation: {seen:,}/{len(loader.dataset):,}", flush=True)
    data.synchronize(device)
    elapsed = time.perf_counter() - start
    reports = {}
    for c in candidates:
        reports[c] = data.report_from_matrices(matrices[c])
        reports[c].update(candidate=c, inference_seconds=elapsed, seconds_per_pulse=elapsed / seen,
                          timing_scope="Loader, I/Q conversion, FFT, template scoring, class reduction and metric accumulation")
    audit_reports = {c: data.report_from_matrices(audit[c]) for c in candidates} if collect_audit else {}
    return reports, audit_reports


def verify_bundle(bundle):
    bundle = Path(bundle).resolve()
    hashes = json.loads((bundle / "frozen.sha256.json").read_text())
    required = {"bank.pt", "config.json", "radarml_track2_bank.py", "radarml_track2_runtime.py",
                "validation_report.json", "requirements.txt"}
    if not required.issubset(hashes):
        raise ValueError("Incomplete Fourier-bank bundle")
    for name, expected in hashes.items():
        if Path(name).name != name or data.lookup_sha256(bundle / name) != expected:
            raise ValueError(f"Frozen artifact changed: {name}")
    if (data.lookup_sha256(Path(__file__)) != hashes["radarml_track2_bank.py"] or
            data.lookup_sha256(Path(data.__file__)) != hashes["radarml_track2_runtime.py"]):
        raise ValueError("Run the code inside the frozen bundle")
    return hashes


def train_and_freeze(root, manifest, run, config=None, kind="auto", prepared=None):
    cfg = json.loads(json.dumps(DEFAULT_CONFIG if config is None else config))
    if (not 1 <= cfg["train_pulses_per_file"] <= 80 or
            not 1 <= cfg["selection_pulses_per_file"] < 20 or
            cfg["batch_size"] < 1 or cfg["workers"] < 0 or
            not cfg["candidates"] or not set(cfg["candidates"]).issubset(CANDIDATES) or
            not 0 <= cfg["window"][0] < cfg["window"][1] <= 16800):
        raise ValueError("Invalid training configuration")
    run, manifest = Path(run), Path(manifest)
    frozen = run / "frozen"
    if (frozen / "frozen.sha256.json").exists():
        verify_bundle(frozen)
        print("Already frozen; reusing completed run:", frozen, flush=True)
        return json.loads((frozen / "validation_report.json").read_text())
    run.mkdir(parents=True, exist_ok=True)
    state = dict(config=cfg, manifest_sha256=data.lookup_sha256(manifest), data_format=kind)
    state_path = run / "run_settings.json"
    if state_path.exists() and json.loads(state_path.read_text()) != state:
        raise ValueError("Run settings changed; choose a new RUN_NAME")
    data.write_json(state_path, state)
    data.seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tick = time.perf_counter()
    rows, infos = prepared if prepared is not None else prepare_development(root, manifest, kind)
    setup_seconds = time.perf_counter() - tick
    # Assert separation before ANY prototype estimation.
    training_ids = {(r["file"], r["pulse_index"]) for r in rows if r["split"] == "train"}
    validation_ids = {(r["file"], r["pulse_index"]) for r in rows if r["split"] == "validation"}
    if training_ids & validation_ids:
        raise ValueError("Source-pulse leakage")
    # Bind resumable training caches to their dataset identity (not Kaggle mount paths).
    identity = {}
    for name, info in infos.items():
        meta_path = Path(info["path"]).with_suffix(".json")
        # resolve_metadata has already checked source/array pairings in prepare_development;
        # the array hash also handles Kaggle's renamed FSK files correctly.
        identity[name] = data.lookup_sha256(Path(info["path"]))
    identity_path = run / "training_data_hashes.json"
    if identity_path.exists() and json.loads(identity_path.read_text()) != identity:
        raise ValueError("Training dataset changed since the cached fit")
    data.write_json(identity_path, identity)
    bank, fit_report = fit_bank(rows, infos, run / "training_cache", cfg)
    model = FourierBank(bank).to(device).eval()
    selection_ids = selected_identities(rows, cfg["selection_pulses_per_file"])
    examples = data.build_examples(rows, infos, "validation", cfg["selection_pulses_per_file"], ALL_SNRS)
    loader = data.make_loader(DevelopmentDataset(examples, infos), cfg["batch_size"], cfg["workers"], cfg["seed"])
    reports, _ = evaluate_candidates(model, loader, device, cfg["candidates"])
    best = cfg["candidates"][0]
    for candidate in cfg["candidates"][1:]:
        score, old = reports[candidate], reports[best]
        delta = score["mean_balanced_accuracy"] - old["mean_balanced_accuracy"]
        if delta > 1e-6 or (abs(delta) <= 1e-6 and score["lowest_three_mean_balanced_accuracy"] > old["lowest_three_mean_balanced_accuracy"] + 1e-6):
            best = candidate
    data.write_json(run / "selection_candidates.json", reports)
    print("Selected on validation only:", best, reports[best]["mean_balanced_accuracy"], flush=True)
    examples = data.build_examples(rows, infos, "validation", 20, ALL_SNRS)
    loader = data.make_loader(DevelopmentDataset(examples, infos, selection_ids), cfg["batch_size"], cfg["workers"], cfg["seed"])
    full, audit = evaluate_candidates(model, loader, device, [best], collect_audit=True)
    report = full[best]
    report.update(full_published_validation=True, pulses_per_recording=20,
                  selection_pulses_per_recording=cfg["selection_pulses_per_file"])
    audit[best].update(candidate=best, excludes_selection_source_pulses=True)
    cfg.update(track="B", track_number=2, model_family="complex_fft_template_bank", classes=list(CLASSES),
               selected_candidate=best, inference_precision="float32", inference_augmentation="none",
               manifest_sha256=data.MANIFEST_SHA256, hardware=data.hardware_info(device),
               actual_data_format=next(iter(infos.values()))["kind"], setup_seconds=setup_seconds,
               model_parameters=int(bank["templates"].numel() * 2),
               external_training_data=False, pretrained=False)
    frozen.mkdir(parents=True, exist_ok=True)
    torch.save(bank, frozen / "bank.pt")
    cfg["model_bytes"] = (frozen / "bank.pt").stat().st_size
    data.write_json(frozen / "config.json", cfg)
    data.write_json(frozen / "fit_report.json", fit_report)
    data.write_json(frozen / "selection_candidates.json", reports)
    data.write_json(frozen / "selection_validation.json", reports[best])
    data.write_json(frozen / "validation_report.json", report)
    data.write_json(frozen / "audit_validation.json", audit[best])
    shutil.copy2(manifest, frozen / "split_manifest.csv")
    shutil.copy2(identity_path, frozen / "training_data_hashes.json")
    for source in (Path(__file__), Path(data.__file__)):
        target = frozen / source.name
        if source.resolve() != target.resolve():
            shutil.copy2(source, target)
    (frozen / "requirements.txt").write_text(f"numpy=={np.__version__}\ntorch=={str(torch.__version__).split('+')[0]}\n")
    (frozen / "inference_command.txt").write_text(
        "python radarml_track2_bank.py infer --bundle . --blind-root /path/to/participants --output /path/to/submission.csv\n")
    summary = f"""RadarML Lite Track 2 / B: learned complex Fourier template classifier

Training: {fit_report['source_pulses']:,} published training pulses only, seed {cfg['seed']}.
Template source: {fit_report['training_snr']}. {fit_report['templates']} complex mean waveforms,
with phase alignment only for training sets whose phase lock is below 0.98.
Fixed acquisition window: samples {cfg['window']}. No filename-derived cropping.
Inference: unitary complex FFT, training-template inner products, per-pulse noise
normalization; selected likelihood/class reduction: {best}.
The coherent_clip score attenuates the expected template by 1 minus the observed
per-pulse clipping fraction. Mixture pooling uses equal class priors.
The eight alternatives are fixed analytic scoring rules, not eight neural networks.
Only validation chooses the scoring rule; learned template parameters use train only.

Selection: first {cfg['selection_pulses_per_file']} published validation indices per recording.
Full validation ({report['samples']:,}): {report['mean_balanced_accuracy']:.8f} mean balanced accuracy.
Separate validation audit excluding selection pulses: {audit[best]['mean_balanced_accuracy']:.8f}.
Fit time: {fit_report['fit_seconds']:.2f} s, excluding dataset verification and validation.
Validation inference including FFT: {report['seconds_per_pulse']:.6f} s/pulse.
Model size: {cfg['model_bytes']:,} bytes. Hardware and versions: config.json.

Limitations: specifically evaluates known configurations with repeatable acquisition
phase and timing. Accuracy on new configurations, phase rotations, time offsets or
hardware is not established. Clipping correction is an approximation. Validation
is development evidence, not a blind score. No 0.995 leaderboard guarantee is made.
This is a new experiment; if the event's model freeze has passed, organizer approval
is needed to replace the previously frozen entry. No blind signals enter this notebook's training.

Attribution: Ritchie, White and Hosford (2025), RadarML, https://doi.org/10.5522/04/30752767.v1.
Organizer helpers adapted under GPL-3.0. AI coding assistance: OpenAI Codex.
Method reference: https://www.mathworks.com/help/signal/ug/radar-pulse-compression.html
Implementation reference: https://docs.pytorch.org/docs/stable/generated/torch.fft.fft.html
"""
    (frozen / "method_summary.txt").write_text(summary, encoding="utf-8")
    hashes = {p.name: data.lookup_sha256(p) for p in sorted(frozen.iterdir()) if p.is_file() and p.name != "frozen.sha256.json"}
    data.write_json(frozen / "frozen.sha256.json", hashes)
    print("FROZEN BUNDLE:", frozen, "| full validation:", report["mean_balanced_accuracy"], flush=True)
    return report


def predict_blind(bundle, blind_root, output, batch_size=128, workers=2):
    bundle, output = Path(bundle).resolve(), Path(output).resolve()
    hashes = verify_bundle(bundle)
    if output.is_relative_to(bundle):
        raise ValueError("Keep generated predictions outside the frozen bundle")
    cfg = json.loads((bundle / "config.json").read_text())
    if cfg["selected_candidate"] not in CANDIDATES or cfg["inference_precision"] != "float32":
        raise ValueError("Invalid frozen inference settings")
    data.seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(bundle / "bank.pt", map_location="cpu", weights_only=True)
    model = FourierBank(checkpoint).to(device).eval()
    del checkpoint
    root, rows, template_ids = data.read_blind_index(blind_root)
    rows.sort(key=lambda r: (r["file"], r["row"]))
    loader = data.make_loader(data.BlindDataset(root, rows), batch_size, workers, cfg["seed"])
    predictions = []
    data.synchronize(device)
    start = time.perf_counter()
    with torch.inference_mode():
        for step, signals in enumerate(loader, 1):
            logits = model(signals.to(device, non_blocking=True), cfg["selected_candidate"])
            predictions.extend(logits.argmax(1).cpu().tolist())
            if step == 1 or step % 100 == 0:
                print(f"Blind inference: {len(predictions):,}/{len(rows):,}", flush=True)
    data.synchronize(device)
    elapsed = time.perf_counter() - start
    if len(predictions) != len(rows):
        raise RuntimeError("Incomplete inference")
    by_id = {r["sample_id"]: CLASSES[p] for r, p in zip(rows, predictions)}
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_name(output.name + ".partial")
    with temporary.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["sample_id", "predicted_label"])
        writer.writerows((sid, by_id[sid]) for sid in template_ids)
    with temporary.open(newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        saved = list(reader)
        if (reader.fieldnames != ["sample_id", "predicted_label"] or
                [r["sample_id"] for r in saved] != template_ids or
                any(r["predicted_label"] not in CLASSES for r in saved)):
            raise ValueError("Serialized submission is invalid")
    temporary.replace(output)
    report = dict(samples=len(rows), inference_seconds=elapsed, seconds_per_pulse=elapsed / len(rows),
                  precision="float32", selected_candidate=cfg["selected_candidate"],
                  timing_scope="Cold loader startup, disk reads, transfers, FFT, template likelihood and prediction collection",
                  hardware=data.hardware_info(device), frozen_artifact_hashes=hashes,
                  predictions_sha256=data.lookup_sha256(output), batch_size=batch_size, workers=workers)
    data.write_json(output.with_name(output.name + ".inference.json"), report)
    print("Submission:", output, flush=True)
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    infer = commands.add_parser("infer")
    infer.add_argument("--bundle", required=True)
    infer.add_argument("--blind-root", required=True)
    infer.add_argument("--output", required=True)
    infer.add_argument("--batch-size", type=int, default=128)
    infer.add_argument("--workers", type=int, default=2)
    args = parser.parse_args()
    if args.batch_size < 1 or args.workers < 0:
        parser.error("batch-size must be positive and workers nonnegative")
    predict_blind(args.bundle, args.blind_root, args.output, args.batch_size, args.workers)


if __name__ == "__main__":
    main()


## Configure fitting and validation

All **80 training source pulses per recording** fit each complex mean template.
Compact inputs use the original recorded training pulses. Expanded inputs use only
the **+30 dB training variants** for fitting, to avoid averaging synthetic noise into
the templates. Every training source pulse retains its published assignment.

The fixed acquisition window is samples **2400:14640** (20–122 µs), identical for
all templates and unrelated to filenames. The noise estimate uses samples 0:2400.
Template construction preserves stable observed phase. For training groups with phase
lock below 0.98, it aligns training pulses before averaging; the phase-invariant
alternative can then be selected by validation. There is no blind fitting or averaging.

Each input is transformed to a full **complex FFT**, preserving phase. Eight fixed
scoring alternatives compare it with the learned bank: known phase, known phase with
an approximate clipping correction, marginalized unknown phase, or free amplitude;
each uses either maximum likelihood or an equal-class-prior template mixture.
The scoring alternatives share one bank; they are not eight neural networks.

The first four published validation indices per recording, at all 13 SNRs, select
the rule using the organizer's mean balanced accuracy, with the specified low-SNR
tie-break. Validation never updates template values or learned calibration.
The selected rule then receives full validation, plus an audit excluding those
selection source pulses. Do not choose a new rule using the audit report.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
sys.path.insert(0, str(Path.cwd()))
import radarml_track2_runtime as helpers
import radarml_track2_bank as bank
helpers = importlib.reload(helpers)
bank = importlib.reload(bank)
torch.set_num_threads(4)

CONFIG = copy.deepcopy(bank.DEFAULT_CONFIG)
CONFIG.update(batch_size=BATCH_SIZE, workers=WORKERS)
# Leave the published 80/20 source split intact. Do not alter CONFIG when resuming.
# Use a new RUN_NAME for a genuinely new experiment.

if TRAIN_MODEL:
    if not DATA_ROOT.is_dir():
        raise FileNotFoundError(f"Update DATA_ROOT to your development dataset: {DATA_ROOT}")
    manifest = helpers.find_manifest(DATA_ROOT, MANIFEST_PATH)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(json.dumps(helpers.hardware_info(device), indent=2))
    print(json.dumps(CONFIG, indent=2))
    if device.type == "cpu":
        print("Enable a GPU for faster FFT/template scoring. CPU execution is supported.")
    print("Manifest:", manifest)
else:
    print("Only the frozen runtime will execute inference.")


## Fit, select, validate and export

Progress is saved after **each recording's fitted template**. If this cell is
interrupted, rerun it with the same settings to reuse completed templates. If GPU
memory is insufficient, start a new RUN_NAME with a smaller batch size, or retain
settings and free GPU memory before resuming. Template fitting is followed by
34,320-example selection and 171,600-example full validation.

The runtime verifies input identity hashes before reusing cached training parameters.
Reading the expanded arrays for hashing can take several minutes. It memory-maps
the data and does not load all recordings into RAM. Full FP32 complex operations
are used; mixed precision is deliberately avoided in coherent likelihood scoring.

The final freeze directory contains `bank.pt`, executable inference code, settings,
dependency versions, training provenance, selection results, full/audit validation,
confusion matrices, a method summary and SHA256 hashes. A completed frozen run is
verified and reused on rerun. The earlier `radar-1.ipynb` model remains separate.

In [ ]:
if TRAIN_MODEL:
    try:
        full_report = bank.train_and_freeze(DATA_ROOT, manifest, RUN_DIR, CONFIG, kind=DATA_FORMAT)
    except KeyboardInterrupt:
        print("Interrupted. Completed training templates are saved; rerun this cell to resume.")
        resume_zip = shutil.make_archive(str(RUN_DIR) + "_resume", "zip", root_dir=RUN_DIR)
        print("Download this before ending the session:", resume_zip)
        raise
    frozen = RUN_DIR / "frozen"
    frozen_zip = shutil.make_archive(str(RUN_DIR) + "_frozen", "zip", root_dir=frozen)
    print("DOWNLOAD FROZEN MODEL:", frozen_zip)
    print("ZIP SHA256:", helpers.lookup_sha256(Path(frozen_zip)))
    from IPython.display import FileLink, display
    display(FileLink(str(Path(frozen_zip).relative_to(WORK_ROOT))))


## Inspect measured validation evidence

The full score is the mean of class-balanced accuracy at **all 13 requested SNRs**.
Class counts are unequal; ordinary accuracy is a different metric. Reports contain
raw confusion matrices (true rows, predicted columns), per-class recall and requested
SNR curves. The separate audit uses 16 validation source pulses per recording,
excluding the four used for model selection. It is still development validation.

In [ ]:
if TRAIN_MODEL:
    frozen = RUN_DIR / "frozen"
    full = json.loads((frozen / "validation_report.json").read_text())
    audit = json.loads((frozen / "audit_validation.json").read_text())
    candidates = json.loads((frozen / "selection_candidates.json").read_text())
    assert full["samples"] == 171600 and audit["samples"] == 137280
    print("Candidate selection results (validation subset):")
    for name, value in sorted(candidates.items(), key=lambda item: -item[1]["mean_balanced_accuracy"]):
        print(f"  {name:26s} {value['mean_balanced_accuracy']:.8f}")
    print("Selected:", full["candidate"])
    print(f"FULL VALIDATION: {full['mean_balanced_accuracy']:.8f}")
    print(f"SEPARATE AUDIT:  {audit['mean_balanced_accuracy']:.8f}")
    print(f"Lowest three SNRs: {full['lowest_three_mean_balanced_accuracy']:.8f}")
    print("Requested SNR | balanced accuracy | ordinary accuracy")
    for row in full["per_snr"]:
        print(f"{row['requested_snr_db']:+3d} dB | {row['balanced_accuracy']:.6f} | {row['accuracy']:.6f}")
    plot_dir = RUN_DIR / "plots"
    plot_dir.mkdir(exist_ok=True)
    fig, ax = plt.subplots(figsize=(9, 4))
    for label, report in [("Full validation", full), ("Audit excluding selection", audit)]:
        ax.plot([r["requested_snr_db"] for r in report["per_snr"]],
                [r["balanced_accuracy"] for r in report["per_snr"]], "o-", label=label)
    ax.axhline(0.995, color="gray", linestyle="--", label="0.995 reference")
    ax.set(xlabel="Requested SNR (dB)", ylabel="Seven-class balanced accuracy", ylim=(0, 1.01))
    ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
    fig.savefig(plot_dir / "validation_snr_curve.png", dpi=160)
    plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for ax, snr in zip(axes, (-20, -25, -30)):
        row = next(r for r in full["per_snr"] if r["requested_snr_db"] == snr)
        matrix = np.array(row["confusion_matrix"], dtype=float)
        ax.imshow(matrix / matrix.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
        ax.set(title=f"Requested {snr:+d} dB", xlabel="Predicted", ylabel="True")
        ax.set_xticks(range(7), helpers.CLASSES, rotation=45, ha="right")
        ax.set_yticks(range(7), helpers.CLASSES)
    fig.tight_layout(); fig.savefig(plot_dir / "low_snr_confusions.png", dpi=160)
    plt.show()
    print((frozen / "method_summary.txt").read_text())


## Save a resumable archive

Run this cell before stopping a session. Upload the extracted archive as a private
Kaggle dataset, set `RESUME_FROM` to its root, and retain the same run name/settings.
A Kaggle session killed without saving its output may lose its working directory;
the notebook cannot recover files that Kaggle did not retain. Use **Save Version**
to retain output artifacts. The frozen archive is the one needed for inference.

In [ ]:
if TRAIN_MODEL:
    resume_zip = shutil.make_archive(str(RUN_DIR) + "_resume", "zip", root_dir=RUN_DIR)
    print("Resume archive:", resume_zip)
    from IPython.display import FileLink, display
    display(FileLink(str(Path(resume_zip).relative_to(WORK_ROOT))))


## Submission inference from the frozen bank

**Train and submit in one Kaggle run (default):** leave `MODE="train_and_submit"`
and attach the released participant package as a Kaggle dataset. This cell uses
`RUN_DIR / "frozen"`, the model produced above, and automatically locates the
participant package under `/kaggle/input`, including nested `participants` folders.
Participant signals are first opened here, after fitting, selection and freezing.
If multiple packages are attached, set `BLIND_ROOT` explicitly in the settings cell.

**Submit using an already trained model:** use `MODE="infer"` and point
`FROZEN_BUNDLE` at its extracted frozen directory. Attach the participant dataset;
training data is unnecessary. This mode skips fitting and validation.

**Training only:** use `MODE="train"` to skip submission generation.

This cell checks every frozen artifact hash **before executing the saved code**.
It checks all 171,600 IDs, follows template order, and writes
`submission_track2.csv` with exactly `sample_id,predicted_label`, plus a hardware,
timing and provenance report. All signal processing is included in inference timing.
It cannot compute a blind score without the organizers' private labels.

**Use this notebook or the new bundle's command for the new model.** The previous
`status/generate_submission.py` belongs to the old CNN bundle and does not load this
new bank. Standalone inference, from the extracted new frozen directory:

```bash
python radarml_track2_bank.py infer --bundle . --blind-root /path/to/participants \
  --output /path/to/submission_track2.csv
```

In [ ]:
def find_participants_root(explicit=None, search_root=Path("/kaggle/input")):
    """Find a unique package containing its template and anonymous index."""
    root = Path(explicit if explicit is not None else search_root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Participant dataset directory does not exist: {root}")
    # An explicit path may point directly to the test/ directory.
    if explicit is not None and (root / "index.csv").is_file():
        if (root / "submission_template.csv").is_file():
            return root
        if (root.parent / "submission_template.csv").is_file():
            return root.parent
    candidates = set()
    for template in root.rglob("submission_template.csv"):
        package = template.parent
        if (package / "test/index.csv").is_file() or (package / "index.csv").is_file():
            candidates.add(package.resolve())
    if not candidates:
        raise FileNotFoundError(
            f"No participant package found under {root}. Attach the extracted participants "
            "dataset with submission_template.csv and test/index.csv, or set BLIND_ROOT."
        )
    if len(candidates) > 1:
        choices = "\n".join(str(path) for path in sorted(candidates))
        raise ValueError(f"Multiple participant packages found; set BLIND_ROOT to one:\n{choices}")
    return candidates.pop()


if GENERATE_SUBMISSION:
    frozen = (RUN_DIR / "frozen" if TRAIN_MODEL else Path(FROZEN_BUNDLE)).resolve()
    hashes = json.loads((frozen / "frozen.sha256.json").read_text())
    required = {"bank.pt", "config.json", "radarml_track2_bank.py", "radarml_track2_runtime.py"}
    if not required.issubset(hashes):
        raise ValueError("Incomplete frozen bank bundle")
    for name, expected in hashes.items():
        if Path(name).name != name:
            raise ValueError("Invalid artifact path")
        actual = hashlib.sha256((frozen / name).read_bytes()).hexdigest()
        if actual != expected:
            raise ValueError(f"Frozen artifact changed: {name}")
    participants = find_participants_root(BLIND_ROOT)
    print("Frozen model:", frozen)
    print("Participant package:", participants)
    subprocess.run([sys.executable, str(frozen / "radarml_track2_bank.py"), "infer",
                    "--bundle", str(frozen), "--blind-root", str(participants),
                    "--output", str(PREDICTIONS_PATH), "--batch-size", str(BATCH_SIZE),
                    "--workers", str(WORKERS)], check=True)
    print(json.loads(PREDICTIONS_PATH.with_name(PREDICTIONS_PATH.name + ".inference.json").read_text()))
    from IPython.display import FileLink, display
    display(FileLink(str(PREDICTIONS_PATH.relative_to(WORK_ROOT))))
    display(FileLink(str(PREDICTIONS_PATH.with_name(PREDICTIONS_PATH.name + ".inference.json").relative_to(WORK_ROOT))))
    print("Download your submission CSV:", PREDICTIONS_PATH)
else:
    print("Submission generation skipped because MODE='train'.")


## Interpretation and references

This approach is tailored to **known waveform configurations and repeatable capture
phase/timing**. A high score here does not establish performance on new waveforms,
receivers, time offsets or arbitrary phase rotations. The phase-invariant alternatives
are included for validation comparison; the coherent option may be strongest for
this acquisition protocol. Requested SNR differs from achieved SNR, and low-SNR int16
clipping changes the noise distribution. The clipping correction is approximate.

No training-source filename, index, requested-SNR tag or metadata enters prediction.
Training grouping manages the supplied recordings and target families; the fitted
weights are waveform estimates. Inference uses only each input's I/Q-derived Fourier
coefficients, fixed acquisition constants and the frozen training-template bank.
No validation/test pulse is used as a template. No archive search recovers blind labels.

- Ritchie, White and Hosford (2025), [RadarML](https://doi.org/10.5522/04/30752767.v1).
- [Radar pulse compression / matched filtering](https://www.mathworks.com/help/signal/ug/radar-pulse-compression.html).
- [PyTorch complex FFT documentation](https://docs.pytorch.org/docs/stable/generated/torch.fft.fft.html).
- Supplied `hackathon/EVENT_RULES.md`, loader, scorer and compact-data expander.
- Code reuse: organizer GPL-3.0 helpers and the prior local runtime. AI assistance: OpenAI Codex.